# Moyenne masquée : séparer somme, compte et division

Retrouver une moyenne indépendante du remplissage.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#moyennes-masquees).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous reprenons exactement le lot du chapitre sur le padding. La percussion n’a qu’un instant réel : son remplissage ne doit pas diviser son niveau apparent par deux.

| Extrait | Somme des deux descripteurs | Nombre d’instants valides | Moyenne |
| --- | --- | --- | --- |
| Voix | [0,3 ; 4,0] | 2 | [0,15 ; 2,0] |
| Percussion | [0,2 ; 4,5] | 1 | [0,2 ; 4,5] |

Chaque descripteur garde sa propre somme. Le compte temporel est partagé sur les deux descripteurs du même extrait.

Le masque est de forme (2,2) : extraits puis temps. Pour multiplier un tableau (2,2,2), nous ajoutons à ce masque un dernier axe de taille un.

## Démonstration 1 · Voir le numérateur et le dénominateur

Décomposer l’expression plutôt que cacher trois calculs dans une ligne.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

La dernière ligne [0,0] appartient au remplissage. Le masque, construit avec les longueurs, est faux uniquement à cette position.

In [4]:
batch = torch.tensor([[[0.1, 1.5], [0.2, 2.5]],
                      [[0.2, 4.5], [0., 0.]]])
masque = torch.tensor([[True, True], [True, False]])


In [5]:
observer('batch', batch)
observer('masque', masque)

batch forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.10000000149011612, 1.5], [0.20000000298023224, 2.5]], [[0.20000000298023224, 4.5], [0.0, 0.0]]]
masque forme (2, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, False]]


### Calculer l’état suivant

unsqueeze(-1) insère le dernier axe. Une validité temporelle peut ainsi multiplier les deux descripteurs, sans copier manuellement le masque.

In [6]:
validite = masque.unsqueeze(-1)


In [7]:
observer('validite', validite)

validite forme (2, 2, 1) type torch.bool requires_grad False
  valeurs : [[[True], [True]], [[True], [False]]]


### Calculer l’état suivant

La multiplication annule les cases non valides. sum(dim=1) supprime l’axe temporel et garde extraits puis descripteurs.

In [8]:
valeurs_valides = batch * validite
sommes = valeurs_valides.sum(dim=1)


In [9]:
observer('valeurs_valides', valeurs_valides)
observer('sommes', sommes)

valeurs_valides forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.10000000149011612, 1.5], [0.20000000298023224, 2.5]], [[0.20000000298023224, 4.5], [0.0, 0.0]]]
sommes forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.30000001192092896, 4.0], [0.20000000298023224, 4.5]]


### Calculer l’état suivant

Le compte est (2,1), avec les valeurs 2 et 1. La division le diffuse sur les deux colonnes de chaque extrait.

La ligne de voix est divisée par 2. Celle de percussion est divisée par 1, même si le rectangle contient deux instants.

In [10]:
comptes = validite.sum(dim=1)
moyennes = sommes / comptes


In [11]:
observer('comptes', comptes)
observer('moyennes', moyennes)

comptes forme (2, 1) type torch.int64 requires_grad False
  valeurs : [[2], [1]]
moyennes forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.15000000596046448, 2.0], [0.20000000298023224, 4.5]]


### Calculer l’état suivant

mean(dim=1) compte toutes les lignes du rectangle. Pour la percussion, il moyenne la ligne réelle avec [0,0].

In [12]:
naives = batch.mean(dim=1)


In [13]:
observer('naives', naives)

naives forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.15000000596046448, 2.0], [0.10000000149011612, 2.25]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("masque etendu :", tuple(validite.shape))
print("sommes :", sommes)
print("comptes :", comptes.tolist())
print("moyennes :", moyennes)


masque etendu : (2, 2, 1)
sommes : tensor([[0.3000, 4.0000],
        [0.2000, 4.5000]])
comptes : [[2], [1]]
moyennes : tensor([[0.1500, 2.0000],
        [0.2000, 4.5000]])


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [15]:
print("moyennes naives :", naives)



moyennes naives : tensor([[0.1500, 2.0000],
        [0.1000, 2.2500]])


### Calculer l’état suivant

Cette seconde expérience est une MSE illustrative des descripteurs contre zéro. Elle ne mesure ni une prédiction du modèle ni une reconstruction sonore.

In [16]:
cible_zero = torch.zeros_like(batch)
erreurs = batch - cible_zero
carres_valides = erreurs.square() * validite


In [17]:
observer('cible_zero', cible_zero)
observer('erreurs', erreurs)
observer('carres_valides', carres_valides)

cible_zero forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]]
erreurs forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.10000000149011612, 1.5], [0.20000000298023224, 2.5]], [[0.20000000298023224, 4.5], [0.0, 0.0]]]
carres_valides forme (2, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[0.010000000707805157, 2.25], [0.04000000283122063, 6.25]], [[0.04000000283122063, 20.25], [0.0, 0.0]]]


### Calculer l’état suivant

Il y a trois instants valides et deux descripteurs par instant. Le numérateur vaut 28,84 et le dénominateur 6.

In [18]:
numerateur = carres_valides.sum()
positions_valides = masque.sum()
descripteurs_par_position = batch.shape[-1]
denominateur = positions_valides * descripteurs_par_position


In [19]:
observer('numerateur', numerateur)
observer('positions_valides', positions_valides)
observer('descripteurs_par_position', descripteurs_par_position)
observer('denominateur', denominateur)

numerateur forme () type torch.float32 requires_grad False
  valeurs : 28.84000015258789
positions_valides forme () type torch.int64 requires_grad False
  valeurs : 3
descripteurs_par_position = 2
denominateur forme () type torch.int64 requires_grad False
  valeurs : 6


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [20]:
perte = numerateur / denominateur


In [21]:
observer('perte', perte)

perte forme () type torch.float32 requires_grad False
  valeurs : 4.806666851043701


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [22]:
print(f"MSE vers zero : {numerateur.item():.2f} / {denominateur.item()} = {perte.item():.6f}")

MSE vers zero : 28.84 / 6 = 4.806667


Une moyenne dépend de ce qui est compté : ici, d’abord les instants par extrait, puis toutes les valeurs pour la MSE.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez la moyenne de [2,4,0,0] quand seules les deux premières positions sont réelles.

In [23]:
exo_x = torch.tensor([2.,4.,0.,0.])
exo_valide = torch.tensor([True,True,False,False])
exo_moyenne = ...  # Remplacer les points par votre réponse.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le numérateur vaut 6 et le nombre d’observations vaut 2.

</details>

### Exercice 2

Changez seulement les deux valeurs de padding en 9 et 20. Quel résumé masqué obtient-on ?

In [24]:
exo_x = torch.tensor([2.,4.,9.,20.])
exo_valide = torch.tensor([True,True,False,False])
exo_moyenne = ...  # Remplacer les points par votre réponse.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les positions invalides ne deviennent pas des observations lorsque leur valeur change.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([2.,4.,0.,0.])
exo_valide = torch.tensor([True,True,False,False])
exo_moyenne = 3.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([2.,4.,9.,20.])
exo_valide = torch.tensor([True,True,False,False])
exo_moyenne = 3.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-exo_x[exo_valide].mean().item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [25]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()